# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inju7/flyrank-ai-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!git clone https://github.com/inju7/flyrank-ai-ml.git /content/flyrank-ai-ml

Cloning into '/content/flyrank-ai-ml'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 123 (delta 35), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 1.86 MiB | 12.24 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [5]:
from pathlib import Path

data_path = Path(
    "/content/flyrank-ai-ml/data/raw/content_refresh_anonymized.csv"
)

print("Dataset exists:", data_path.exists())
print("Repository exists:", Path("/content/flyrank-ai-ml").exists())

Dataset exists: True
Repository exists: True


In [6]:
import pandas as pd

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring, which I frame as a ranking task. The goal is to prioritize content pages for human review by ordering them according to their potential need for attention. A content editor or SEO specialist can then review the highest-priority pages and decide whether to refresh, expand, protect, prune, monitor, or leave them unchanged. Ranking is appropriate because the goal is to identify which pages should be reviewed first, rather than simply classify every page as good or bad. The available dataset provides signals for building an initial ranking, but the ranking must be evaluated before it can be trusted for decision-making.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Dataset shape:", df.shape)
print("Number of unique content items:", df["content_id"].nunique())
print("Available columns:", len(df.columns))

Dataset shape: (30000, 44)
Number of unique content items: 30000
Available columns: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The ideal target would be an observed outcome showing whether a content page benefitted from editiorial attention, such as an improvement in performance after a refresh. However, the started dataset does not directly provide that outcome. For this initial task, I will use is_declining_label as a proxy target, defined as 1 when trend_direction is down and 0 otherwise. This proxy identifies that refreshing it iwll improve performance. Because this lavel is derived from trend_direction and ultimately trend_pct, I must exclude both fields from the predictive features to avoid taret leakage. The model's results will therefore indicate how well it identifies the proxy, not whether it predicts successful content refreshes.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype("int8")
)

print("Proxy target distribution:")
print(df["is_declining_label"].value_counts().sort_index())

print("\nProxy target prevalence:")
print(f"{df['is_declining_label'].mean():.1%}")

Proxy target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Proxy target prevalence:
54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary evaluation metric is Precision@50, the proportion of the 50 highest-ranked pages that have is_declining_label = 1. For example, a Precision@50 of 0.50 would mean that 25 of the top 50 ranked pages are labeled as declining. This is an illustrative example, not a measured result. A useful ranking should achieve a higher Precision@50 than a simple baseline evaluated on the same held-out data. The baseline should be compared fairly, with leakage-safe validation and the same evaluation set. Since the target is only declining-trend proxy, a good score would show that the ranking prioritizes pages labeled as declining: it would not prove that those pages are the best refresh opportunities.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
declining_rate = df["is_declining_label"].mean()

print(f"Overall declining-label prevalence: {declining_rate:.1%}")
print(
    "Expected declining pages among 50 under random ranking:",
    round(50 * declining_rate, 2)
)

Overall declining-label prevalence: 54.2%
Expected declining pages among 50 under random ranking: 27.1


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one pseudonymized content item per row. Each row represents a content page with attributes and performance signals, such as content age, impressions, clicks, and average search position. The content_id identifies the item for inspection and grouping, but it should not be used as a predictive feature. I will use the available content attributes and performance signals to explore how pages differ and prepare a ranking. The proxy target is derived from the existing trend category, not from a measured outcome following a content refresh.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from IPython.display import display

columns_to_show = [
    "content_id",
    "content_type",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "is_declining_label",
]

display(df[columns_to_show].head(10))

print("Total rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("\nRows per content type:")
print(df["content_type"].value_counts(dropna=False))

,content_id,content_type,content_age_days,impressions_90d,clicks_90d,avg_position,is_declining_label
0,content_304f48230142,keyword article,187,3803,29,10.6,1
1,content_a1fb4e703a9e,keyword article,445,15320,7,20.3,1
2,content_9aa793d4d895,keyword article,141,12581,11,36.5,1
3,content_331d6c4de07b,keyword article,463,11751,58,6.2,0
4,content_d99b7a2d90ca,keyword article,263,19140,24,44.0,1
5,content_d4084a4bc775,keyword article,147,3970,1,8.5,1
6,content_9a34b442b552,keyword article,90,20,0,7.0,1
7,content_a63219c6e95a,keyword article,445,1724,1,21.2,0
8,content_5e6c160719bc,keyword article,90,32574,29,46.0,1
9,content_c27558df2b0c,keyword article,257,1240,2,4.9,1


Total rows: 30000
Unique content IDs: 30000

Rows per content type:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule, such as prioritizing pages older than a chosen number of days or pages with declining impressions, would provide a simple and explainable baseline. However, content priority may depend on several signals together, including content age, impressions, clicks, search position, and engagement. A single threshold may overlook interactions between these signals or rank very different pages similarly. Machine learning could learn combinations of signals and produce a more useful ordering, but it is not automatically better than a fixed rule. I would compare an ML ranking against a transparent rule-based baseline using the same leakage-safe evaluation process and Precision@50. If ML does not provide a meaningful improvement, or if its results are difficult to explain, the simpler rule may be preferable. Any improvement on the current dataset would apply to the declining-label proxy, not proven refresh success.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
candidate_features = [
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
]

available_features = [
    col for col in candidate_features if col in df.columns
]

print("Candidate predictive signals:")
print(available_features)

print("\nMissing values in candidate signals:")
print(df[available_features].isna().sum())

print("\nFields excluded to prevent target leakage:")
print(["trend_direction", "trend_pct", "content_id", "client_id"])

Candidate predictive signals:
['content_age_days', 'impressions_90d', 'clicks_90d', 'avg_position']

Missing values in candidate signals:
content_age_days    0
impressions_90d     0
clicks_90d          0
avg_position        0
dtype: int64

Fields excluded to prevent target leakage:
['trend_direction', 'trend_pct', 'content_id', 'client_id']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.